# 9d. Production Caches : payer une fois, servir mille fois

Deuxième volet production après [9c_Production_Routage_Repli](09c_Production_Routage_Repli.ipynb)
(choisir le bon modèle) : ici, **ne pas rappeler le modèle du tout**. Un appel LLM coûté une fois
peut resservir — à condition de savoir *quand* réutiliser sans se tromper.

Deux familles de caches, mesurées sur le terrain réel :

1. **Le cache sémantique** — reconnaître qu'une question a déjà été posée *à un mot près*,
   par similarité d'embeddings. Sa mesure critique : le **taux de faux succès** — des questions
   presque identiques dont la réponse est différente, servies depuis le cache comme si de rien
   n'était ;
2. **Le cache de préfixe fournisseur** — OpenAI facture moins cher les tokens de prompt déjà
   vus (`cached_tokens` dans la réponse). On mesure ce que ça rend vraiment, et à quelles
   conditions l'allonger se justifie.

> **Terrain réel, lecture seule** : embeddings OpenAI (`text-embedding-3-small`, clé dans `.env`,
> jamais affichée), catalogue de prix du proxy multi-fournisseurs (`/v1/models`), inférence locale
> Ollama pour les réponses à mettre en cache. Rien de tout cela n'est reconfiguré.


Comment ce notebook se lit : chaque mesure a sa sortie brute et, collée à sa suite, sa **lecture** — la cellule qui n'existe que pour elle ; les introductions de section précèdent la mesure et restent en prospective. Le fil : trois caches de natures différentes (exact, sémantique, préfixe fournisseur), chacun mesuré sur le même terrain que le 9c, chacun avec sa monnaie (millisecondes, faux succès, dollars) — et à la fin, la question qui les départage : qu'achète chacun, et que risque-t-on ?

## Vérification de l'environnement


Même protocole d'ouverture que le 9c : on sonde le terrain en lecture seule avant de toucher quoi que ce soit — ce qui suit configure l'inférence, mais la santé du terrain se constate d'abord. Une jambe morte ici se verrait immédiatement, pas au milieu d'une mesure.

In [1]:
import os, json, time, random, re, statistics, math
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
print("OPENAI_API_KEY     :", "présente" if OPENAI_API_KEY else "ABSENTE -- sections embeddings/préfixe impossibles")
print("OPENROUTER_API_KEY :", "présente" if OPENROUTER_API_KEY else "absente")

random.seed(20260928)   # même graine que 9c : même banque


OPENAI_API_KEY     : présente
OPENROUTER_API_KEY : présente


In [2]:
# Mêmes sondages lecture seule que 9c : on re-confirme le terrain, on ne le configure pas
import urllib.request
import pandas as pd
from openai import OpenAI

def sonde(nom, url):
    try:
        with urllib.request.urlopen(urllib.request.Request(url), timeout=15) as r:
            n = len(json.loads(r.read().decode("utf-8")).get("data", []))
        print(f"{nom:38s} HTTP 200  {n:4d} modèles")
    except Exception as e:
        print(f"{nom:38s} INJOIGNABLE ({type(e).__name__})")

sonde("Inference locale (Ollama)", "http://127.0.0.1:11434/v1/models")
sonde("Proxy multi-fournisseurs", "https://openrouter.ai/api/v1/models")

def client_openai():
    assert OPENAI_API_KEY, "OPENAI_API_KEY absente du .env"
    return OpenAI(api_key=OPENAI_API_KEY)

def client_local():
    return OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")

# Prix réels du proxy pour les modèles de ce notebook (lus au moment de l'exécution)
req = urllib.request.Request("https://openrouter.ai/api/v1/models")
with urllib.request.urlopen(req, timeout=15) as r:
    catalogue = {m["id"]: m for m in json.loads(r.read().decode("utf-8"))["data"]}

MODELE_CHAT = "openai/gpt-5-mini"
assert MODELE_CHAT in catalogue, f"{MODELE_CHAT} absent du catalogue proxy"
PRIX_IN = float(catalogue[MODELE_CHAT]["pricing"]["prompt"]) * 1e6
PRIX_OUT = float(catalogue[MODELE_CHAT]["pricing"]["completion"]) * 1e6
print(f"Prix catalogue {MODELE_CHAT} : ${PRIX_IN:.2f}/M entrée, ${PRIX_OUT:.2f}/M sortie")


Inference locale (Ollama)              HTTP 200     2 modèles
Proxy multi-fournisseurs               HTTP 200   464 modèles


Prix catalogue openai/gpt-5-mini : $0.25/M entrée, $2.00/M sortie


### Lecture — le terrain : un modèle local, un catalogue de 464, un tarif étalon

Deux terrains confirmés en lecture seule : l'inférence locale Ollama répond (**HTTP 200, 2 modèles** chargés) et le proxy multi-fournisseurs expose **464 modèles** — le notebook vivra sur ces deux jambes, le local pour remplir le cache, le catalogue pour les prix. Le prix étalon retenu pour la route mesurée : **openai/gpt-5-mini à $0,25/M entrée, $2,00/M sortie**.

Ces sondages ne configurent rien (aucune clé utilisée ici) : ils établissent que le terrain du 9c est intact — même Ollama, même proxy — pour que les comparaisons entre les deux notebooks portent sur les caches, jamais sur une infrastructure qui aurait bougé.

## 1. La banque de 9c, et ses pièges

Le cache sémantique se juge sur deux populations adverses, pas sur des répétitions bénignes :

- la **banque** de questions arithmétiques (même générateur, même graine que 9c) ;
- les **pièges** : mêmes phrases, **nombres différents** — « 320 colis puis 540 » contre
  « 320 colis puis 541 ». Leurs embeddings sont quasi jumeaux, leurs réponses diffèrent.
  Un cache qui sert le piège depuis l'entrée voisine fabrique un **faux succès**.

C'est la population qui manque toujours aux démos de cache sémantique — et celle qui décide
du seuil en production.


Pourquoi des pièges plutôt qu'un simple jeu de test : un cache ne se juge pas sur ce qu'il sert bien, mais sur ce qu'il sert **tort**. Une question piège est la jumelle exacte d'une question cachée, à un détail près qui change la réponse — le dispositif minimal qui transforme « taux de hit » en une métrique honnête.

In [3]:
def q_addition(a, b):
    return f"Un entrepôt reçoit {a} colis le matin, puis {b} colis l'après-midi. Combien de colis au total ?", a + b

def q_soustraction(a, b):
    return f"Un magasin possède {a} unités de stock et en vend {b} dans la journée. Combien reste-t-il d'unités ?", a - b

def q_produit(a, b):
    return f"Une imprimerie produit {a} affiches par série et fabrique {b} séries. Combien d'affiches au total ?", a * b

GENES = [("addition", q_addition), ("soustraction", q_soustraction), ("produit", q_produit)]

rnd = random.Random(20260928)
BANQUE = []
for i in range(90):   # 30 de chaque type
    nom, gene = GENES[i % 3]
    if nom == "addition":
        texte, rep = gene(rnd.randint(120, 980), rnd.randint(120, 980))
    elif nom == "soustraction":
        texte, rep = gene(rnd.randint(400, 990), rnd.randint(100, 390))
    else:
        texte, rep = gene(rnd.randint(12, 96), rnd.randint(12, 96))
    BANQUE.append({"id": i, "type": nom, "question": texte, "reponse": rep})

# Pièges : même phrase, second opérande décalé de +1 -> réponse différente, embedding jumeau
PIEGES = []
for i, q in enumerate(BANQUE[:30]):
    a, b = (int(n) for n in re.findall(r"\d+", q["question"])[:2])
    gene = dict(GENES)[q["type"]]
    texte, rep = gene(a, b + 1)
    PIEGES.append({"id": i, "type": q["type"], "question": texte, "reponse": rep})

print(f"Banque : {len(BANQUE)} questions | Pièges : {len(PIEGES)} (second opérande +1)")
for q in (BANQUE[0], PIEGES[0]):
    print(f"  [{q['type']:11s}] {q['question']}  ->  {q['reponse']}")


Banque : 90 questions | Pièges : 30 (second opérande +1)
  [addition   ] Un entrepôt reçoit 823 colis le matin, puis 682 colis l'après-midi. Combien de colis au total ?  ->  1505
  [addition   ] Un entrepôt reçoit 823 colis le matin, puis 683 colis l'après-midi. Combien de colis au total ?  ->  1506


### Lecture — la banque et ses pièges : 1505 contre 1506, à un token d'écart

**90 questions** générées (30 additions, 30 soustractions, 30 produits) et **30 pièges** : la même question que sa jumelle de banque, avec le **second opérande incrémenté de 1**. L'exemple affiché le montre crûment : « 823 puis 682 colis → 1505 » jouxte « 823 puis 683 colis → 1506 ». Un cache qui répond depuis la similarité verra ces deux questions comme presque identiques — alors que la bonne réponse diffère.

C'est le dispositif expérimental du cache sémantique entier : la banque mesure ce que le cache **économise** (requêtes légitimes servies sans appel), les pièges mesurent ce qu'il **coûte** (faux succès — une réponse juste pour la mauvaise question). Les deux populations sont adverses par construction ; aucune métrique qui les sépare n'est honnête.

## 2. Remplir le cache : une passe réelle sur l'inférence locale

Le cache ne stocke pas des réponses plaquées : on fait vraiment répondre le modèle local aux 90
questions de la banque (gratuit, température 0), et ce sont **ces réponses réelles** qui
entrent au cache. Les pièges, eux, ne sont jamais mis en cache — ils arrivent comme des
requêtes clientes. Cette passe fournit aussi la **latence de référence** d'un appel non caché,
qui servira de dénominateur à toutes les économies mesurées ensuite.


Un choix de protocole important : le cache se remplit avec les **réponses réelles** du modèle local, pas avec les réponses calculées par le générateur. La différence compte pour tout ce qui suit — ce que le cache servira, c'est le modèle tel qu'il répond, erreurs comprises ; on ne peut pas mesurer la dégradation d'un pipeline sans d'abord mesurer son fond.

In [4]:
LOCAL_ID = "qwen2.5:7b"
SYSTEME = ("Tu es un assistant de calcul. Réponds en une seule phrase courte et termine par la "
           "valeur numérique exacte, sans unité.")

def extraire_nombre(texte):
    nombres = re.findall(r"-?\d+", texte or "")
    return int(nombres[-1]) if nombres else None

def poser_local(question):
    t0 = time.perf_counter()
    rep = client_local().chat.completions.create(
        model=LOCAL_ID,
        messages=[{"role": "system", "content": SYSTEME},
                  {"role": "user", "content": question}],
        temperature=0.0, max_tokens=200,
    )
    dt = time.perf_counter() - t0
    texte = rep.choices[0].message.content or ""
    return extraire_nombre(texte), dt

CACHE = []   # [{question, reponse_modele, correcte}]
latences = []
t0 = time.perf_counter()
for q in BANQUE:
    pred, dt = poser_local(q["question"])
    latences.append(dt)
    CACHE.append({**q, "reponse_modele": pred, "correcte": pred == q["reponse"]})
duree_remplissage = time.perf_counter() - t0

n_justes = sum(c["correcte"] for c in CACHE)
print(f"Cache rempli : {len(CACHE)} réponses réelles du modèle local en {duree_remplissage:.0f} s")
print(f"qualité du fond de cache : {n_justes}/{len(CACHE)} justes ({100.0 * n_justes / len(CACHE):.0f} %)")
LAT_REF = statistics.median(latences)
print(f"latence de référence d'un appel non caché (médiane) : {LAT_REF * 1000:.0f} ms")


Cache rempli : 90 réponses réelles du modèle local en 74 s
qualité du fond de cache : 78/90 justes (87 %)
latence de référence d'un appel non caché (médiane) : 806 ms


### Lecture — le fond du cache : 87 % juste, 806 ms par appel frais

Le cache est rempli par **90 réponses réelles** du modèle local en 74 s — pas de réponses synthétiques : ce qui sera servi plus tard est ce que le modèle a réellement dit. Deux nombres portent toute la suite :

- **78/90 justes (87 %)** : le fond de cache lui-même n'est pas parfait. Un hit ne garantit pas une réponse exacte — il garantit la réponse **du modèle à cette question-là**. Le plafond d'exactitude du pipeline caché est 87 % avant même le premier faux succès.
- **806 ms de latence médiane** par appel non caché : c'est l'unité de compte de l'économie. Quand le balayage du §5 dit « 88,7 s évités », ce sont 88,7 s **à ce débit-là**, mesuré sur le même terrain au même moment.

## 3. Embeddings réels et similarité : à quoi ressemble un « presque pareil »

Toutes les questions — fond de cache **et** pièges — partent en embeddings réels chez OpenAI
(`text-embedding-3-small`, 1536 dimensions, par lots). Avant tout réglage, on **regarde** la
distribution : la similarité cosinus entre chaque piège et l'entrée la plus proche du cache,
séparée en deux populations — la jumelle (même question, autre nombre) et les questions
réellement différentes. Si les deux distributions se chevauchent, aucun seuil ne sera propre —
et c'est le premier résultat du notebook.


L'outillage est minimal et réel : des embeddings API (`text-embedding-3-small`), une similarité cosine, et l'entrée la plus proche dans le cache. Toute la question du seuil γ se jouera dans l'espace que cette cellule mesure — les jumelles d'un côté, les étrangères de l'autre, et l'écart entre les deux.

In [5]:
def embedder(textes, lot=64):
    vecteurs = []
    for i in range(0, len(textes), lot):
        r = client_openai().embeddings.create(model="text-embedding-3-small",
                                               input=textes[i:i + lot])
        vecteurs.extend(d.embedding for d in r.data)
    return vecteurs

emb_cache = embedder([c["question"] for c in CACHE])
emb_pieges = embedder([p["question"] for p in PIEGES])

def cos(u, v):
    dot = sum(x * y for x, y in zip(u, v))
    nu = math.sqrt(sum(x * x for x in u))
    nv = math.sqrt(sum(x * x for x in v))
    return dot / (nu * nv) if nu and nv else 0.0

def plus_proche(vect):
    best_i, best_s = 0, -1.0
    for i, u in enumerate(emb_cache):
        s = cos(vect, u)
        if s > best_s:
            best_i, best_s = i, s
    return best_i, best_s

# Population 1 : la jumelle (même gabarit, autre nombre). Résultat de terrain attendu :
# l'entrée la plus proche d'un piège est systématiquement sa jumelle.
sim_jumelles, rang_jumelle = [], 0
for j, v in enumerate(emb_pieges):
    i, s = plus_proche(v)
    sim_jumelles.append(s)
    if i == j:
        rang_jumelle += 1

# Population 2 (contraste) : pour chaque piège, la similarité de la question « étrangère »
# la plus proche — un autre GABARIT, que n'importe quel seuil raisonnable doit repousser.
sim_etrangeres = [
    max(cos(v, u) for i2, u in enumerate(emb_cache) if CACHE[i2]["type"] != p["type"])
    for p, v in zip(PIEGES, emb_pieges)
]

print(f"la plus proche entrée du cache est la jumelle : {rang_jumelle}/{len(PIEGES)} pièges")
print(f"piège -> jumelle   : min {min(sim_jumelles):.3f}  moy {statistics.mean(sim_jumelles):.3f}  max {max(sim_jumelles):.3f}  (n={len(sim_jumelles)})")
print(f"piège -> étrangère : min {min(sim_etrangeres):.3f}  moy {statistics.mean(sim_etrangeres):.3f}  max {max(sim_etrangeres):.3f}  (n={len(sim_etrangeres)})")


la plus proche entrée du cache est la jumelle : 30/30 pièges
piège -> jumelle   : min 0.962  moy 0.986  max 0.996  (n=30)
piège -> étrangère : min 0.457  moy 0.550  max 0.612  (n=30)


### Lecture — l'écart jumelle/étrangère : 0,986 contre 0,550, sans recouvrement

Les similarités cosine (embeddings `text-embedding-3-small` réels) séparent nettement deux populations :

- **piège → jumelle** : min 0,962, moy 0,986, max 0,996 — et 30 fois sur 30, l'entrée la plus proche d'un piège **est** sa jumelle ;
- **piège → étrangère** : min 0,457, moy 0,550, max 0,612.

L'écart entre les deux maxima (0,996 contre 0,612) est immense mais **asymétrique en pratique** : il n'existe aucun seuil au-dessus de 0,612 qui sépare un piège de sa jumelle, précisément parce qu'ils sont faits pour être inséparables. Toute la dramatique du §5 est déjà là : le seuil qui élimine les faux succès (proche de 1) ne peut pas toucher les paires jumelles — il ne peut que refuser de servir, ce qui coûte des hits.

## 4. Le cache exact : le plancher des droits

Avant le sémantique, l'exact : une clé de dict sur le texte de la question. Re-poser la banque
ne coûte qu'une recherche dans un dict ; re-poser une **reformulation** (« Quel est le nombre total de colis ? ») coûte un
appel complet. Le cache exact est gratuit et sans risque — sa mesure fixe le plancher que le
sémantique doit dépasser **sans** payer en faux succès.


Le cache exact est le **référentiel absolu** de la famille : zéro approximation, zéro faux succès, coût dict. Il ne sert pas à remplacer les deux autres — il délimite ce qu'ils ajoutent. Ce que cette section mesure, c'est sa frontière exacte : là où la chaîne de caractères cesse d'être identique, le service cesse.

In [6]:
cache_exact = {c["question"]: c["reponse_modele"] for c in CACHE}

t0 = time.perf_counter()
hits = sum(1 for q in BANQUE if q["question"] in cache_exact)
dt_exact = time.perf_counter() - t0
print(f"rejeu de la banque par cache exact : {hits}/{len(BANQUE)} hits, {dt_exact * 1000:.1f} ms au total")

def reformuler(q):
    return (q["question"]
            .replace("Combien de colis au total ?", "Quel est le nombre total de colis ?")
            .replace("Combien d'affiches au total ?", "Quel est le nombre total d'affiches ?")
            .replace("Combien reste-t-il d'unités ?", "Quel est le nombre d'unités restantes ?"))

reformulations = [reformuler(q) for q in BANQUE[:20]]
assert all(r != q["question"] for r, q in zip(reformulations, BANQUE[:20])), "reformulation identique"
hits_ref = sum(1 for r in reformulations if r in cache_exact)
print(f"reformulations (n=20) : {hits_ref} hit(s) — le cache exact est aveugle à la formulation")


rejeu de la banque par cache exact : 90/90 hits, 0.2 ms au total
reformulations (n=20) : 0 hit(s) — le cache exact est aveugle à la formulation


### Lecture — le cache exact : 0,2 ms, 90/90… et aveugle hors du verbatim

Le plancher des droits est écrasant : rejouer les 90 questions par simple dict coûte **0,2 ms au total** — quatre ordres de grandeur sous l'appel à 806 ms — avec 100 % de hits sur les questions **identiques au byte près**.

Et la limite tombe aussitôt : les 20 reformulations (« Combien de colis au total ? » → « Quel est le nombre total de colis ? ») font **0 hit**. Une seule tournure change, le sens est identique, le cache exact ne voit rien. C'est la définition même de sa frontière : gratuit et parfait **sur la chaîne exacte**, muet dès la première variation de surface — le vide que le cache sémantique du §5 vient remplir, à son prix.

## 5. Le cache sémantique : le seuil et son prix

Le cache sémantique répond depuis l'entrée la plus proche dès que la similarité dépasse un
seuil γ. On balaie γ : à chaque seuil, on rejoue trois populations — la banque (le hit est
légitime), les reformulations (le hit est le gain), et les pièges (le hit est une **fausse
réponse servie**). Les réponses « fraîches » hors seuil sont calculées **une seule fois** par
question puis mémoïsées : le balayage fait varier le seuil, pas le terrain.

Trois courbes à lire ensemble : économie de latence, taux de hit, et taux de faux succès. Le
seuil retenu en production est celui où la courbe de faux succès tombe au niveau que le
produit tolère — pas celui qui maximise l'économie.


Le protocole du balayage tient en une phrase : pour chaque seuil γ, servir depuis le cache toute requête dont la similarité à l'entrée la plus proche dépasse γ, sinon appeler le modèle — et compter séparément banque servie (hits), pièges servis (faux succès) et exactitude de bout en bout. Trois populations, une seule table, et une lecture à faire dans l'ordre : similarité jumelle, faux succès par seuil, exactitude.

In [7]:
requetes = ([(q["question"], q["reponse"], "banque") for q in BANQUE[:60]]
            + [(reformuler(q), q["reponse"], "reformulation") for q in BANQUE[:20]]
            + [(p["question"], p["reponse"], "piege") for p in PIEGES])
emb_requetes = embedder([r[0] for r in requetes])

# Réponses fraîches mémoïsées : chaque question hors seuil est réellement posée au modèle
# la première fois seulement ; les seuils suivants rejouent la même réalité.
frais = {}
def reponse_fraiche(question):
    if question not in frais:
        frais[question] = poser_local(question)[0]
    return frais[question]

def rejouer_avec_seuil(seuil):
    """Renvoie (hits, faux_succes, justes) pour un seuil gamma donné.

    faux_succes = un PIEGE servi DEPUIS LE CACHE avec la mauvaise reponse.
    Une erreur de reponse fraiche (le modele local se trompe) n'est pas un
    faux succes : elle reste visible dans la colonne exactitude."""
    hits = faux = justes = 0
    for (question, reponse, famille), v in zip(requetes, emb_requetes):
        i, s = plus_proche(v)
        if s >= seuil:
            hits += 1
            pred = CACHE[i]["reponse_modele"]
            if pred != reponse and famille == "piege":
                faux += 1
        else:
            pred = reponse_fraiche(question)
        if pred == reponse:
            justes += 1
    return hits, faux, justes

mesures = []
for seuil in [0.90, 0.95, 0.98, 0.99, 0.995, 0.999]:
    hits, faux, justes = rejouer_avec_seuil(seuil)
    mesures.append({
        "seuil γ": seuil,
        "hits": hits,
        "taux de hit %": round(100.0 * hits / len(requetes), 1),
        "faux succès (pièges)": faux,
        "faux succès %": round(100.0 * faux / len(PIEGES), 1),
        "exactitude %": round(100.0 * justes / len(requetes), 1),
        f"latence évitée (s, à {LAT_REF * 1000:.0f} ms/appel)": round(hits * LAT_REF, 1),
    })
print(f"{len(frais)} appels frais réels au total (mémoïsés) ; le balayage lui-même est de l'arithmétique")
df_seuils = pd.DataFrame(mesures).set_index("seuil γ")
df_seuils


50 appels frais réels au total (mémoïsés) ; le balayage lui-même est de l'arithmétique


,hits,taux de hit %,faux succès (pièges),faux succès %,exactitude %,"latence évitée (s, à 806 ms/appel)"
seuil γ,,,,,,
0.900,110,100.0,30,100.0,63.6,88.7
0.950,110,100.0,30,100.0,63.6,88.7
0.980,98,89.1,21,70.0,71.8,79.0
0.990,72,65.5,12,40.0,75.5,58.0
0.995,61,55.5,1,3.3,81.8,49.2
0.999,60,54.5,0,0.0,82.7,48.4


### Lecture du balayage — un seuil sûr existe, et il coûte un tiers des hits

Le tableau se lit en trois temps, et les nombres cette fois y répondent :

1. **La similarité jumelle (§3) bornait tout** : pièges et jumelles à 0,96-1,00. Effectivement, à γ = 0,900 et 0,950, le cache sert **100 % des requêtes — y compris les 30 pièges, faux succès 100 %**. Le faux succès n'est pas un accident de réglage : il est structurel tant que le seuil est sous la similarité jumelle.
2. **Un seuil sûr existe** : à γ = 0,995, il ne reste qu'**1 faux succès sur 30** (3,3 %) avec encore 55,5 % de hits ; à γ = 0,999, **zéro faux succès**, 54,5 % de hits. Le cache sémantique n'est pas disqualifié sur ce domaine — mais le seuil sûr rend un service deux fois plus petit que le seuil naïf (54,5 % contre 100 %).
3. **Le prix se paie en exactitude et en latence rendue** : l'exactitude du pipeline monte de **63,6 %** (seuils lâches, les pièges servis) à **82,7 %** (γ = 0,999) ; la latence évitée tombe de **88,7 s à 48,4 s** sur la série. Chaque point d'exactitude acheté coûte des hits.

Note de méthode : les 50 appels frais sont mémoïsés — le balayage rejoue la même réalité pour chaque seuil ; les colonnes sont donc comparables entre elles, ce sont les mêmes réponses qui changent de source.

## 6. Le cache de préfixe fournisseur : lire `cached_tokens`

Chez OpenAI, un préfixe de prompt déjà vu (au-delà d'un seuil de longueur, 1024 tokens) est
facturé moins cher quand il revient : la réponse expose
`usage.prompt_tokens_details.cached_tokens`. Le terrain dit la vérité — on ne la devine pas.
On construit un **préfixe métier long** (règles de calcul et conventions du service logistique),
on pose une série de questions courtes qui le partagent, et on lit les compteurs de chaque
réponse.


Troisième cache, troisième nature : les deux premiers vivent **chez nous** (un dict, un index d'embeddings) ; celui-ci vit **chez le fournisseur** — OpenAI facture un préfixe de prompt déjà vu à une fraction du prix d'entrée. Rien à construire, rien à calibrer : la seule mesure utile est de voir la mécanique à l'œuvre dans `cached_tokens`, puis de la chiffrer aux tarifs du catalogue.

In [8]:
PREFIXE = (
    "Tu es l'assistant de calcul du service logistique Aurelia, rattaché à la direction des "
    "opérations. Tu réponds aux questions de quantités (colis, unités de stock, affiches) en "
    "appliquant strictement les règles du service. Règle 1 : les totaux incluent toujours les "
    "livraisons du matin ET celles de l'après-midi, sans exception, même si l'après-midi est "
    "partiellement reporté au lendemain. Règle 2 : les retours clients sont déduits du stock "
    "après la vente, jamais avant la mise en rayon ; un retour non contrôlé reste en stock. "
    "Règle 3 : les pourcentages s'appliquent au total brut, arrondis à l'entier inférieur, et "
    "une remise en cascade se compose multiplicativement, jamais par addition. Règle 4 : les "
    "séries de production se multiplient, les rebuts se soustraient en fin de série, et une "
    "série interrompue compte pour son dernier lot complet. Règle 5 : un colis prioritaire "
    "est un colis ; la priorité ne change jamais les quantités, seulement l'ordre de départ. "
    "Règle 6 : les doublons de réception (même numéro de bon livré deux fois) comptent une "
    "seule fois dans les totaux du jour. Règle 7 : une rupture de stock ramène le disponible à "
    "zéro, jamais à un nombre négatif. Règle 8 : les conversions d'unités internes (palettes, "
    "cartons, unités) utilisent la table officielle du service, palettes de 120, cartons de 12, "
    "et ne se devinent jamais. Règle 9 : en cas d'incohérence entre deux chiffres de l'énoncé, "
    "le chiffre horodaté le plus récent fait foi. Règle 10 : les reports inter-entrepôts ne "
    "comptent ni comme sorties ni comme entrées du jour concerné. Conventions de réponse : "
    "répondre en une phrase courte, terminer par la valeur numérique exacte, ne jamais "
    "afficher le détail des opérations intermédiaires, ne jamais reformuler la question, ne "
    "jamais ajouter d'unité, ne jamais commenter les règles elles-mêmes. Contexte du service : "
    "les rapports trimestriels des trois dernières années ont toujours validé ces règles par "
    "audit ; l'assistant n'a pas à les justifier, seulement à les appliquer. Domaine couvert : "
    "entrepôts régionaux, stocks de magasins, séries d'impression, dossiers prioritaires, "
    "réceptions du matin et de l'après-midi, retours clients, conversions palettes et cartons. "
    "Règle 11 : les stocks d'ouverture annoncés deux fois dans un énoncé ne se comptent qu'une "
    "fois ; la seconde mention est une confirmation, pas une entrée supplémentaire. Règle 12 : "
    "les arrondis de conversion palettes versent le reste en unités libres, jamais en palette "
    "incomplète facturée pleine. Règle 13 : un transfert partiellelement livré compte les "
    "quantités effectivement parties, à la date de départ effective du camion. Règle 14 : les "
    "tolérances de pesée (plus ou moins une unité par carton) s'annulent en moyenne sur les "
    "totaux d'une série complète. Règle 15 : en fin de mois, les reports non soldés du mois "
    "précédent s'ajoutent aux entrées du premier jour ouvré, sans recalcul rétroactif. "
    "Règle 16 : une commande annulée avant départ n'a jamais existé dans aucun total. "
    "Glossaire du service : un lot est un groupe de cartons d'une même référence ; une "
    "réception est un événement horodaté de déchargement ; un disponible est le stock moins "
    "les réservations confirmées ; une réserve n'est jamais un disponible. Procédure de "
    "contrôle : après chaque réponse, l'assistant vérifie mentalement l'ordre de grandeur ; si "
    "le résultat dépasse dix mille unités pour une question de magasin, ou descend sous "
    "zéro, l'assistant recompte silencieusement avant de répondre, sans jamais mentionner ce "
    "contrôle. Traçabilité : les réponses alimentent le journal d'exploitation quotidien ; "
    "elles doivent donc rester autoportantes, sans reprise du contexte de la question. "
    "Historique de formulation : les questions arrivent courtes ; l'assistant ne demande "
    "jamais d'éclaircissement et prend l'énoncé tel quel. Registre attendu : sobre, direct, "
    "factuel, sans politesse superflue et sans formule d'ouverture ; la réponse commence par "
    "le résultat du calcul, jamais par une salutation. Cas limites déjà tranchés par le "
    "service : une question sans nombre vaut réponse « 0 » ; une question avec trois nombres "
    "applique les règles dans l'ordre d'apparition des nombres ; une question répétée dans la "
    "même journée reçoit la même réponse sans variante. Ces décisions ne se discutent pas et "
    "ne se mentionnent pas : elles s'appliquent. Dernier rappel de registre : aucun "
    "préambule, aucune excuse, aucune explication de méthode — le service paie au token de "
    "sortie, chaque mot inutile est une dépense. Manuel de procédure du service, à connaître "
    "et à appliquer sans le citer : avant la première réponse d'une journée, l'assistant "
    "relise mentalement les règles 1 à 16 dans l'ordre ; aucun rapport ne mentionne cette "
    "relecture. Les incidents classés des trois dernières années, par fréquence décroissante : "
    "totaux matin comptés deux fois (règle 1), retours déduits avant vente (règle 2), "
    "pourcentages appliqués au net au lieu du brut (règle 3), séries interrompules comptées "
    "complètes (règle 4), palettes incomplètes facturées pleines (règle 12). Chaque incident "
    "a donné lieu à une correction de règle, jamais à une exception individuelle : il n'existe "
    "aucune dérogation, aucun cas particulier, aucune validation hiérarchique possible. Les "
    "chiffres ronds suspects (multiples de cent) reçoivent la même vérification que les "
    "autres. La réponse finale ne contient jamais le mot « règle », jamais le numéro d'une "
    "règle, jamais une référence à ce manuel.")

def poser_chat(question):
    rep = client_openai().chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "system", "content": PREFIXE},
                  {"role": "user", "content": question}],
        max_completion_tokens=150,   # gpt-5-mini : temperature non reglable (defaut 1)
    )
    u = rep.usage
    details = getattr(u, "prompt_tokens_details", None)
    cache_tok = getattr(details, "cached_tokens", 0) if details is not None else 0
    return {"prompt": u.prompt_tokens, "cached": cache_tok,
            "completion": u.completion_tokens}

# 10 questions partagent le préfixe ; la première écrit le cache du fournisseur.
# Son écriture est asynchrone : sans quelques secondes d'écart, le second appel repart
# aussi frais que le premier.
questions_pref = [q["question"] for q in BANQUE[:10]]
mesures_pref = [poser_chat(questions_pref[0])]
time.sleep(10)
mesures_pref += [poser_chat(q) for q in questions_pref[1:]]
df_pref = pd.DataFrame([{
    "appel": i + 1,
    "tokens prompt": m["prompt"],
    "dont cachés": m["cached"],
    "part cachée %": round(100.0 * m["cached"] / m["prompt"], 1) if m["prompt"] else 0.0,
} for i, m in enumerate(mesures_pref)]).set_index("appel")
df_pref


,tokens prompt,dont cachés,part cachée %
appel,,,
1,1304,0,0.0
2,1305,1152,88.3
3,1304,1152,88.3
4,1304,1152,88.3
5,1305,1152,88.3
6,1304,1152,88.3
7,1304,1152,88.3
8,1305,1152,88.3
9,1304,1152,88.3


### Lecture du cache de préfixe — plein tarif une fois, 88 % servi ensuite

Cette exécution est partie d'un cache **froid** : le 1ᵉʳ appel affiche **0 token caché** sur 1 304 — il paie le préfixe au plein tarif — et dès le 2ᵉ appel, **1 152 tokens sur ~1 304 (88,3 %)** sont servis depuis le cache, et la série ne bouge plus : les appels 2 à 10 affichent exactement la même part. Ce contraste entre la première ligne et les neuf suivantes EST la mécanique du cache fournisseur : un premier paiement plein tarif, puis la même part au tarif réduit à chaque appel (le cache survit d'une exécution à l'autre ; une série démarrée peu après une autre peut afficher son 1ᵉʳ appel déjà chaud — c'est précisément son intérêt en production).

L'économie se chiffre avec les tarifs de la cellule suivante : le token caché facturé à **0,1× le prix d'entrée** ($0,025/M contre $0,25/M sur la route mesurée). Et la bonne question n'est pas un « point d'équilibre » universel : si le préfixe est **commun** aux appels — un contexte qu'on enverrait de toute façon —, chaque hit est une économie immédiate, il n'y a rien à amortir ; si on **ajoute** un préfixe long pour atteindre le cache, il faut le comparer à la requête courte qu'on vient d'allonger — la cellule suivante chiffre les deux scénarios.

In [9]:
# Tarif cache de la route mesurée : gpt-5-mini facture l'entrée CACHÉE à 0,1 fois le prix
# d'entrée (politique publiée par le fournisseur, relevée le 2026-10-01). Ce ratio n'est pas
# universel : il dépend de la route et du modèle -- nommer les deux avant d'estimer une facture.
R_CACHED = 0.1
prix_cache = R_CACHED * PRIX_IN
tokens_prompt = mesures_pref[-1]["prompt"]
tokens_cache = mesures_pref[-1]["cached"]
part_cachee = tokens_cache / tokens_prompt if tokens_prompt else 0.0

def cout_appel(entree_non_cachee, entree_cachee, sortie):
    """Coût exact d'un appel, poste par poste : entrée non cachée, entrée cachée, sortie."""
    return (entree_non_cachee * PRIX_IN + entree_cachee * prix_cache + sortie * PRIX_OUT) / 1e6

print(f"route : {MODELE_CHAT} via proxy, catalogue mesuré ${PRIX_IN:.2f}/M entrée, "
      f"${PRIX_OUT:.2f}/M sortie, entrée cachée ${prix_cache:.3f}/M ({R_CACHED:.0%} de l'entrée)")

# Scénario 1 -- préfixe COMMUN : les appels paieraient ce contexte au plein tarif même sans
# cache ; chaque hit économise la différence plein/réduit sur la part servie. Rien à amortir.
economie_par_hit = tokens_cache * (PRIX_IN - prix_cache) / 1e6
print(f"préfixe : ~{tokens_prompt} tokens dont {100 * part_cachee:.0f} % cachés ; scénario préfixe "
      f"commun : chaque hit économise ${economie_par_hit:.7f} -- gain dès le premier hit, sans seuil")

# Scénario 2 -- préfixe AJOUTÉ : on allonge la requête pour atteindre le cache. Témoin,
# entrée seule : requête courte de 100 tokens contre le même prompt long déjà chaud.
court = cout_appel(100, 0, 0)
long_chaud = cout_appel(tokens_prompt - tokens_cache, tokens_cache, 0)
print(f"scénario préfixe ajouté : requête courte = ${court:.7f}, prompt long déjà chaud = ${long_chaud:.7f}")
print(f"le hit fonctionne (entrée servie au tarif réduit) mais la stratégie reste "
      f"${long_chaud - court:.7f} plus chère : un préfixe ajouté ne se justifie que s'il "
      "remplace du travail, jamais pour le cache seul")


route : openai/gpt-5-mini via proxy, catalogue mesuré $0.25/M entrée, $2.00/M sortie, entrée cachée $0.025/M (10% de l'entrée)
préfixe : ~1304 tokens dont 88 % cachés ; scénario préfixe commun : chaque hit économise $0.0002592 -- gain dès le premier hit, sans seuil
scénario préfixe ajouté : requête courte = $0.0000250, prompt long déjà chaud = $0.0000668
le hit fonctionne (entrée servie au tarif réduit) mais la stratégie reste $0.0000418 plus chère : un préfixe ajouté ne se justifie que s'il remplace du travail, jamais pour le cache seul


### Lecture — la facture : le hit paie, le préfixe ajouté ne paie pas

Poste par poste, avec le ratio mesuré (entrée cachée à 10 % du prix d'entrée) :

- **Préfixe commun** (le cas normal : un contexte qu'on enverrait de toute façon) : chaque hit économise **$0,0002592** — le gain est immédiat, dès le premier hit, sans seuil ni amortissement. Servir 88 % d'un prompt de 1 304 tokens au dixième du prix, appel après appel.
- **Préfixe ajouté** (allonger une requête courte pour « atteindre » le cache) : la requête courte coûte $0,0000250 ; la même requête avec préfixe long déjà chaud coûte $0,0000668. Le hit **fonctionne** (l'entrée est servie au tarif réduit) — et pourtant la stratégie reste **$0,0000418 plus chère** que la requête courte.

La leçon tient dans la dernière ligne : un cache récompense ce qu'on faisait déjà, il ne justifie pas ce qu'on fabrique pour lui. Ajouter du contexte pour cibler le cache achète un hit et perd de l'argent — le seul préfixe rentable est celui dont le contenu travaillait déjà.

## 7. Synthèse : ce que chaque cache achète — et ce qu'il risque

| Cache | Ce qu'il économise | Son risque mesuré | Verdict du terrain |
|---|---|---|---|
| exact | 100 % des répétitions à l'identique | aucun | plancher gratuit, toujours actif |
| sémantique | les reformulations (hit mesuré au seuil γ) | **faux succès** : pièges jumeaux servis | selon la distribution mesurée en section 3 |
| préfixe | la part `cached_tokens` du prompt | préfixe payé plein tarif à froid | économie par hit mesurée en section 6 |

La leçon transversale : **un cache est un pari sur la distribution des questions**. Le cache
exact parie qu'elles se répètent à l'identique ; le sémantique qu'elles se répètent *en
forme* sans changer de fond ; le préfixe qu'elles partagent un contexte stable. Ce notebook a
mesuré les trois paris sur le même terrain — avec, pour le sémantique, la population de
pièges qui manque aux démos.

Limites honnêtes : le domaine est arithmétique (les pièges y sont particulièrement jumeaux —
un domaine plus varié espacerait les similarités) ; les embeddings viennent d'un seul
fournisseur ; la latence de référence est celle de l'inférence locale, un appel distant est
plus lent encore et augmente le gain d'un hit.


Comment utiliser cette synthèse : chaque ligne nomme la monnaie dans laquelle le risque se paie — millisecondes pour l'exact, faux succès pour le sémantique, dollars pour le préfixe — et les sections ci-dessus ont mesuré chaque chiffre cité. Le tableau ne remplace pas les mesures : il les range, pour qu'un choix de production se fasse sur la colonne qui correspond au risque qu'on accepte.

## Exercices

### Exercice 1 : cache LRU borné

Le cache de ce notebook croît sans limite. Implémentez un cache LRU borné à N entrées
(N = 50) : mesurez le taux de hit sur un flux où la banque boucle 3 fois — que perd-on
à borner, en hit rate et en faux succès ?

### Exercice 2 : seuil adaptatif par famille

Le seuil γ unique est grossier : la similarité jumelle varie par type de question. Calibrez
un seuil **par famille** (mesuré sur un échantillon de pièges étiquetés) et comparez le couple
(hit rate, faux succès) au seuil unique.

### Exercice 3 : invalidation par TTL

Un cache qui sert des réponses périmées est un faux succès différé. Ajoutez un TTL aux
entrées : une question re-posée après expiration force un appel frais. Mesurez le hit rate
sur un flux avec un TTL court et des re-questions rapprochées.


Les trois exercices suivent l'arc du notebook : le premier outille la **durée de vie** (un cache LRU borné — celui du notebook croît sans limite, la question est bien posée), le deuxième attaque le **seuil** (un γ par famille de questions plutôt qu'un γ global — le balayage du §5 a montré le prix d'un seuil unique), le troisième l'**invalidation** (TTL : jusqu'à quand une réponse cachée reste-t-elle serviable ?). Chacun part des structures déjà en place (`BANQUE`, `CACHE`, `rejouer_avec_seuil`) — le travail est dans la politique de cache, pas dans le replomber.

In [10]:
# Exercice 1 -- cache LRU borné
# TODO étudiant : implémenter CacheLRU(capacite=50) avec get/put et ordre d'usage
# Indice : collections.OrderedDict et move_to_end font le travail en quelques lignes
# Étape 1 : la classe avec move_to_end à chaque get, popitem(last=False) à l'éviction
# Étape 2 : rejouer le flux banque×3 et compter les hits
# Étape 3 : comparer au cache non borné (hits, faux succès)
print("Exercice à compléter")
resultat_ex1 = None  # TODO étudiant : (hits_borne, hits_non_borne)


Exercice à compléter


In [11]:
# Exercice 2 -- seuil adaptatif par famille
# TODO étudiant : calibrer un seuil par type (addition, soustraction, produit)
# Indice : pour chaque famille, la similarité jumelle min mesurée + une marge
# Étape 1 : grouper les mesures de similarité par type
# Étape 2 : seuil_famille = min(sim_jumelles_famille) + marge
# Étape 3 : rejouer le balayage avec seuil par famille et comparer
print("Exercice à compléter")
resultat_ex2 = None  # TODO étudiant : table (famille, seuil, faux succès)


Exercice à compléter


In [12]:
# Exercice 3 -- invalidation par TTL
# TODO étudiant : entrées de cache horodatées, expiration après le TTL choisi
# Indice : time.monotonic() à l'écriture, comparaison à la lecture
# Étape 1 : ajouter "ts" aux entrées du cache
# Étape 2 : get() retourne None si âgé > TTL
# Étape 3 : mesurer le hit rate sur un flux de 90 questions à 1/s avec retours
print("Exercice à compléter")
resultat_ex3 = None  # TODO étudiant : hit rate avec/sans TTL


Exercice à compléter


## Conclusion

### Récapitulatif

| Brique | Question à laquelle elle répond |
|---|---|
| Banque + pièges | « presque pareil » se mesure-t-il ? |
| Fond de cache réel | le cache sert des réponses du modèle, pas plaquées |
| Cache exact | le plancher gratuit |
| Balayage γ | hit rate et faux succès se lisent ENSEMBLE |
| `cached_tokens` | le fournisseur raconte lui-même son cache |

### La suite

Le dernier volet production garde le terrain et ferme la boucle **exploitation** : coût par
requête et par fonctionnalité, budget et alerte, déploiement progressif (shadow → canary →
retour arrière automatique).

### Sources

- Similarité : embeddings `text-embedding-3-small` (1536 dimensions, API réelle)
- Cache de préfixe : compteurs `usage.prompt_tokens_details.cached_tokens` des réponses
- Prix : catalogue `/v1/models` du proxy multi-fournisseurs, lus à l'exécution

## Navigation

**Série GenAI Texte** | Précédent : [9c — Production Routage et Repli](09c_Production_Routage_Repli.ipynb) | Suivant : [9e — Production Exploitation](09e_Production_Exploitation.ipynb) | [↑ Sommaire série](README.md)


Si l'on ne devait retenir qu'une chose : les trois caches ne sont pas trois tailles du même objet, mais trois **négociations différentes** — la certitude absolue sur la chaîne exacte (dict), le pari calibré sur le sens (seuil γ, avec ses faux succès comptés), l'argent immédiat sur le contexte répété (préfixe fournisseur). Chacune a été mesurée sur le même terrain, dans sa propre monnaie ; aucune n'écrase les autres. Le métier, c'est de savoir laquelle répond au risque qu'on a — et de relire la table de synthèse dans cette colonne-là.